# 0 · Data Aggregator
Reads individual stock CSVs from `../data/`, cleans prices, and produces a single wide-format monthly price matrix saved to `../analysis/india_equity_prices.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, '..')
from utils import DATA_DIR, ANALYSIS_DIR

data_path     = Path(DATA_DIR)
analysis_path = Path(ANALYSIS_DIR)
analysis_path.mkdir(parents=True, exist_ok=True)

# ── Load each stock CSV ───────────────────────────────────────────────────────
EXCLUDE = {"nifty"}   # handled separately in nb 6

files = sorted(f for f in data_path.glob("*.csv")
               if f.stem.split()[0].lower() not in EXCLUDE
               and "nifty" not in f.stem.lower())

assert len(files) > 0, f"No CSV files found in {data_path.resolve()}"

dfs = []
for f in files:
    ticker = f.stem.split()[0]          # e.g. 'BRTI Historical Data' → 'BRTI'
    df = pd.read_csv(f)

    # normalise column names
    df.columns = [c.strip() for c in df.columns]

    assert "Date"  in df.columns, f"{f.name}: missing 'Date' column"
    assert "Price" in df.columns, f"{f.name}: missing 'Price' column"

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

    df["Price"] = (
        df["Price"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .astype(float)
    )

    df = df[["Date", "Price"]].rename(columns={"Price": ticker})
    dfs.append(df)
    print(f"  {ticker:6s}  {len(df):4d} rows  {df['Date'].min().date()} → {df['Date'].max().date()}")

# ── Merge on Date (outer so we see all available history) ────────────────────
out = dfs[0]
for df in dfs[1:]:
    out = out.merge(df, on="Date", how="outer")

out = out.sort_values("Date").set_index("Date")

# ── Drop rows where ALL prices are missing ───────────────────────────────────
out = out.dropna(how="all")

# Report any remaining NaNs so the analyst can decide what to do
missing = out.isna().sum()
if missing.any():
    print("\nMissing values per ticker (will be forward-filled):")
    print(missing[missing > 0])

# Forward-fill isolated gaps (≤2 consecutive), then drop any remaining NaN rows
out = out.ffill(limit=2).dropna()

print(f"\nFinal price matrix: {out.shape[0]} months × {out.shape[1]} tickers")
print(f"Date range: {out.index.min().date()} → {out.index.max().date()}")

out.to_csv(analysis_path / "india_equity_prices.csv")
print("\nSaved: analysis/india_equity_prices.csv")
out.head()

  BRTI      86 rows  2019-02-01 → 2026-03-01
  EICH      86 rows  2019-02-01 → 2026-03-01
  HDBK      86 rows  2019-02-01 → 2026-03-01
  HLL       86 rows  2019-02-01 → 2026-03-01
  LART      86 rows  2019-02-01 → 2026-03-01
  RELI      86 rows  2019-02-01 → 2026-03-01
  SUN       86 rows  2019-02-01 → 2026-03-01
  TISC      86 rows  2019-02-01 → 2026-03-01
  TITN      86 rows  2019-02-01 → 2026-03-01

Final price matrix: 86 months × 9 tickers
Date range: 2019-02-01 → 2026-03-01

Saved: analysis/india_equity_prices.csv


,BRTI,EICH,HDBK,HLL,LART,RELI,SUN,TISC,TITN
Date,,,,,,,,,
2019-02-01,286.64,1986.91,516.95,1718.43,1265.54,553.56,445.15,50.04,1025.20
2019-03-01,300.21,2054.77,577.00,1692.79,1355.93,613.01,478.85,52.10,1141.85
2019-04-01,314.40,2036.83,576.64,1743.27,1319.96,626.30,457.65,55.72,1158.55
2019-05-01,342.37,1994.78,603.49,1773.82,1524.53,598.13,409.85,48.83,1235.75
2019-06-01,340.26,1913.88,608.07,1772.93,1520.27,563.48,400.95,50.44,1334.70
